[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-naive-bayes.ipynb)

# Naive Bayes

*AIBits Academy · Machine Learning End To End · Probabilistic Classifier*

A fast, probabilistic classifier based on Bayes' theorem — with a "naive" assumption of feature independence that works surprisingly well for text classification.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

> **🎯 Intuition First**
>
> Think about how a doctor actually reasons through a diagnosis. Before running a single test, they already carry a rough sense of how common a condition is in the population — that's a **prior**. Each new symptom then nudges that belief up or down — each nudge is a **likelihood**. Multiply enough of these nudges together and you land on an updated, evidence-weighted confidence — the **posterior**. Naive Bayes is exactly this process automated: it just adds one simplifying (and admittedly unrealistic) assumption — that every symptom nudges the belief independently of every other symptom — and that shortcut turns out to still work remarkably well in practice.

> **📋 Real-World Case Study — Spam/Ham Email Filtering**
>
> Naive Bayes' most famous production use case is exactly this: classifying email as "spam" or "ham" (legitimate). Words like "free," "winner," and "urgent" carry high P(word | spam), while ordinary vocabulary carries higher P(word | ham) — multiplying these word-level probabilities together (the "naive" independence assumption) is enough to catch the overwhelming majority of spam with very little training data or compute, which is exactly why it remained a mainstay of email filtering for years despite far more sophisticated alternatives being available.

## Bayes' Theorem

Naive Bayes classifies by computing the posterior probability of each class given the input features:

$$P(C\mid x_1,\ldots,x_p) \propto P(C)\cdot \prod_j P(x_j\mid C)$$

> **📊 Prerequisite refresher**
>
> This entire page is one applied example of Bayes' Theorem from the **Probability Fundamentals** prerequisite page — if the posterior/likelihood/prior terminology below feels unfamiliar, that page derives the formula from conditional probability and works through the same base-rate intuition used here.

The **naive** assumption: features are *conditionally independent* given the class. This is rarely true in practice, but makes the computation tractable and often gives competitive results regardless.

## Worked Numeric Example — Full Bayes' Theorem

Suppose 30% of Flipkart reviews are negative (P(neg)=0.3), and the word "poor" appears in 40% of negative reviews but only 5% of positive reviews. For a new review containing "poor":

$$\begin{gathered}P(\text{neg}\mid\text{"poor"}) = \dfrac{P(\text{"poor"}\mid\text{neg})\cdot P(\text{neg})}{P(\text{"poor"}\mid\text{neg})\cdot P(\text{neg}) + P(\text{"poor"}\mid\text{pos})\cdot P(\text{pos})}\\[8pt]= \dfrac{0.40\times 0.30}{0.40\times 0.30 + 0.05\times 0.70} = \dfrac{0.12}{0.155} \approx \mathbf{0.774}\end{gathered}$$

Even though only 30% of reviews are negative overall, seeing "poor" swings the posterior probability of "negative" to 77.4% — this is Bayesian updating: a strong likelihood ratio (0.40 vs 0.05) can overturn a modest prior.

## Try It — Bayes' Theorem Deciding a Class Live

A single-feature version of the Indore triage example: patient temperature (°C), two class-conditional Gaussians (Low risk ~ N(98.4, 0.5²), High risk ~ N(100.8, 0.9²)), with priors P(Low)=0.7, P(High)=0.3. Drag the temperature slider and watch the posterior — computed exactly as P(C|x) ∝ P(C)·P(x|C) — swing between the two classes.

## Laplace (Additive) Smoothing

If a word never appeared in the negative-class training reviews, its raw estimated P(word|neg) = 0 — this would force the *entire* product to zero regardless of how strongly every other word suggests "negative" (a single unseen word veto). Laplace smoothing adds a small pseudo-count α to every word count before normalising:

$$P(x_j\mid C) = \frac{\text{count}(x_j,C)+\alpha}{\text{count}(C)+\alpha\cdot|V|}$$

where |V| is the vocabulary size. α=1 ("add-one smoothing") is the sklearn `MultinomialNB(alpha=1.0)` default seen in the code below — it guarantees every word gets a small nonzero probability under every class, no matter how rare.

## Variants

| Variant | Feature type | P(xⱼ \| C) model | Best for |
|---|---|---|---|
| **GaussianNB** | Continuous | Gaussian (normal) | Sensor data, numerical features |
| **MultinomialNB** | Count / TF-IDF | Multinomial | Text classification, word counts |
| **BernoulliNB** | Binary (0/1) | Bernoulli | Binary features, short texts |
| **ComplementNB** | Count | Complement classes | Imbalanced text datasets |

> **🔬 Why Build It From Scratch At All?**
>
> Re-implementing an algorithm you could simply import is one of the fastest ways to actually understand it — it forces you to confront every term in the formula box above instead of trusting a black box. The discipline that matters isn't the from-scratch code itself, though — it's the verification step right after it: run the identical data through `sklearn`'s own implementation and confirm the numbers agree. If your scratch accuracy and sklearn's accuracy match (as they do below), that's real evidence you understand what the library is doing internally, not just that you know how to call `.fit()`.

## From Scratch — Gaussian Naive Bayes

In [ ]:
# Gaussian Naive Bayes from scratch — Indore hospital disease triage
import numpy as np

class GaussianNB:
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.priors_  = {}
        self.means_   = {}
        self.vars_    = {}
        for c in self.classes_:
            Xc = X[y == c]
            self.priors_[c] = len(Xc) / len(X)
            self.means_[c]  = Xc.mean(axis=0)
            self.vars_[c]   = Xc.var(axis=0) + 1e-9  # epsilon for stability

    def _log_likelihood(self, x, mean, var):
        return -0.5 * np.sum(np.log(2*np.pi*var) + (x-mean)**2/var)

    def predict(self, X):
        preds = []
        for x in X:
            scores = {c: np.log(self.priors_[c]) +
                        self._log_likelihood(x, self.means_[c], self.vars_[c])
                      for c in self.classes_}
            preds.append(max(scores, key=scores.get))
        return np.array(preds)

# Dataset: [age, fever(°C), bp_systolic] → diagnosis (0=Low, 1=High risk)
np.random.seed(12)
n = 300
age   = np.random.normal(45, 12, n)
fever = np.random.normal(99, 1.2, n)
bp    = np.random.normal(120, 18, n)
X = np.column_stack([age, fever, bp])
y = ((age > 50) | (fever > 100) | (bp > 135)).astype(int)

from sklearn.model_selection import train_test_split
X_tr,X_te,y_tr,y_te = train_test_split(X,y,test_size=0.2,random_state=7)

nb = GaussianNB()
nb.fit(X_tr, y_tr)
preds = nb.predict(X_te)
acc = np.mean(preds == y_te)
print(f"Scratch GaussianNB accuracy: {acc:.3f}")

# Compare with sklearn
from sklearn.naive_bayes import GaussianNB as SKLearnNB
sk_nb = SKLearnNB()
sk_nb.fit(X_tr, y_tr)
print(f"sklearn  GaussianNB accuracy: {sk_nb.score(X_te, y_te):.3f}")

## Text Classification — Flipkart Review Sentiment

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

# Sample Flipkart product reviews
reviews = [
    "Superb quality saree, exactly as shown in photo",
    "Very poor stitching, delivered damaged product",
    "Excellent fabric, bright colours, fast delivery",
    "Terrible customer service, wrong size sent",
    "Loved the kurta, fits perfectly, great price",
    "Pathetic quality, faded after first wash",
    "Amazing dupatta set, highly recommend",
    "Waste of money, nothing like the picture",
    "Beautiful lehenga, perfect for wedding season",
    "Cheap material, buttons fell off in a week"
]
labels = [1,0,1,0,1,0,1,0,1,0]  # 1=Positive, 0=Negative

pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=1)),
    ('nb',    MultinomialNB(alpha=1.0))   # alpha = Laplace smoothing
])
pipe.fit(reviews, labels)

new_reviews = [
    "Wonderful product, very happy with purchase",
    "Horrible experience, never buying again"
]
probs = pipe.predict_proba(new_reviews)
for rev, prob in zip(new_reviews, probs):
    print(f"  '{rev[:40]}…'  P(pos)={prob[1]:.3f}")

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Class priors

Store in `priors` a dict mapping each class in `labels` to its relative frequency.

In [ ]:
labels = ["spam", "ham", "ham", "ham", "spam", "ham", "ham", "ham", "ham", "spam"]
priors = None   # TODO


In [ ]:
try:
    check("priors", priors == {"spam": 0.3, "ham": 0.7})
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
labels = ["spam", "ham", "ham", "ham", "spam", "ham", "ham", "ham", "ham", "spam"]
priors = {c: labels.count(c) / len(labels) for c in set(labels)}

```

</details>

### Exercise 2 · Medium · Gaussian likelihood

Write `gauss_pdf(x, mean, var)`, the normal density, and use it to compute the class scores below (prior × likelihood) for a new patient with temperature 38.6. Store the predicted class name in `pred`.

In [ ]:
import numpy as np
def gauss_pdf(x, mean, var):
    pass   # TODO
stats = {"flu": (38.5, 0.3, 0.4), "healthy": (36.8, 0.2, 0.6)}   # class: (mean, var, prior)
pred = None   # TODO


In [ ]:
try:
    from scipy.stats import norm
    check("matches scipy", abs(gauss_pdf(1.0, 0.0, 4.0) - norm.pdf(1.0, 0, 2.0)) < 1e-12)
    check("prediction", pred == "flu")
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
def gauss_pdf(x, mean, var):
    return np.exp(-(x - mean) ** 2 / (2 * var)) / np.sqrt(2 * np.pi * var)
stats = {"flu": (38.5, 0.3, 0.4), "healthy": (36.8, 0.2, 0.6)}
scores = {c: prior * gauss_pdf(38.6, m, v) for c, (m, v, prior) in stats.items()}
pred = max(scores, key=scores.get)

```

</details>

### Exercise 3 · Stretch · A tiny spam filter

Build a `make_pipeline(CountVectorizer(), MultinomialNB())`, fit it on `texts` / `labels`, and store predictions for the two `new` messages in `preds`.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
texts = ["win a free prize now", "free money click now", "meeting at noon tomorrow", "please review the report", "claim your free prize", "lunch tomorrow with the team"]
labels = ["spam", "spam", "ham", "ham", "spam", "ham"]
new = ["free prize inside", "report for the team meeting"]
preds = None   # TODO


In [ ]:
try:
    check("classified correctly", list(preds) == ["spam", "ham"])
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
texts = ["win a free prize now", "free money click now", "meeting at noon tomorrow", "please review the report", "claim your free prize", "lunch tomorrow with the team"]
labels = ["spam", "spam", "ham", "ham", "spam", "ham"]
new = ["free prize inside", "report for the team meeting"]
preds = make_pipeline(CountVectorizer(), MultinomialNB()).fit(texts, labels).predict(new)

```

</details>

---
*Back to the course: **Machine Learning End To End → Naive Bayes**.*